# Walk-Forward Analysis + Monte Carlo

Two robustness layers on top of a backtest, used **in order**:

1. **Parameter sweep + walk-forward (rolling)** — attacks *overfitting*. On each train window the param grid is swept and the best combo is chosen; that combo is then run on the **next, unseen** test window. Stitching the out-of-sample (OOS) test trades together gives a look-ahead-free track record. If the edge only exists in-sample, it dies here.
2. **Monte Carlo on the OOS trades** — attacks *luck / sequence risk*. Block-bootstraps the OOS trade returns into a distribution of terminal return and max drawdown.

Order matters: Monte Carlo on a single tuned in-sample run just launders an overfit edge into a confident-looking distribution. It is run on the **walk-forward OOS** trades, never on the sweep winner.

All three (`sweep`, `walk_forward`, `monte_carlo`) live in [`engine/evaluation.py`](../engine/evaluation.py) and are built on the real `Backtester` + `Trade.pnl_bps` — each OOS segment is an ordinary causal backtest, so no look-ahead is introduced.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import plotly.express as px

from engine.evaluation import sweep, walk_forward, monte_carlo
from engine.strategies import EMACrossoverStrategy
from engine.trade_configurator import ACTIVE_TRADE
from engine.data_configurator import ACTIVE, load_data

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec
# (engine/data_configurator.py) — edit ACTIVE there to change symbol/interval/window.
df = load_data()
SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval
TRADING_CONFIG = ACTIVE_TRADE
print(f"{SYMBOL} {INTERVAL}m — {len(df)} bars from {df.index[0]} to {df.index[-1]}")

# What to evaluate: a strategy, its param grid, and the walk-forward windows.
STRATEGY = EMACrossoverStrategy
GRID = {
    "ema_fast": [5, 9, 13, 17],
    "ema_slow": [20, 30, 40, 50],
}
TRAIN_BARS = 300        # in-sample window swept for the best params
TEST_BARS  = 100        # out-of-sample window the winner is then tested on
OBJECTIVE  = "total_pnl_bps"   # any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
MIN_TRADES = 3          # ignore in-sample combos with fewer trades (noise, not signal)

## 1. Parameter sweep — the single-window (overfit-prone) view

One full-history grid run. This is what a naive optimisation reports — and exactly what walk-forward exists to keep honest. Look at the *region* of good params, not the single peak: a bright cell with bright neighbours is robust; a lone bright cell is usually an overfit outlier.

In [ ]:
sw = sweep(STRATEGY, df, GRID, symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG)
sw[sw.trades >= MIN_TRADES].sort_values("total_pnl_bps", ascending=False).head(8)

In [ ]:
grid_pivot = sw.pivot(index="ema_fast", columns="ema_slow", values="sharpe_approx")
px.imshow(
    grid_pivot, color_continuous_scale="RdYlGn", color_continuous_midpoint=0, aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Sharpe"),
    title=f"In-sample Sharpe — {SYMBOL} {INTERVAL}m (single full-history window)",
).show()

## 2. Walk-forward (rolling) — the honest out-of-sample view

Each `TEST_BARS` window is traded with the params that won the **preceding** `TRAIN_BARS` window's sweep. `WF efficiency (OOS/IS)` ≈ 1 means the in-sample edge carried over; ≪ 1 (or negative) means it was curve-fit. Set `anchored=True` for an expanding (anchored) window instead of rolling.

In [ ]:
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG,
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per-fold: the params chosen in-sample and how they did out-of-sample.
wf.folds_frame()

## 3. Parameter stability

The best params per fold. Columns that jump around fold-to-fold are a red flag — the optimum is chasing noise, so the live-chosen params are unlikely to be the ones that work next window. Stable columns are what you actually trust.

In [ ]:
wf.param_stability()

## 4. Out-of-sample equity curve

The stitched OOS test windows, compounded. This is the closest thing to the equity path you'd have lived through re-tuning periodically on only past data.

In [ ]:
eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
px.line(eq, labels={"value": "equity", "index": ""},
        title=f"Walk-forward OOS equity — {SYMBOL} {INTERVAL}m").update_layout(showlegend=False).show()

## 5. Monte Carlo on the OOS trades — luck & drawdown distribution

Block-bootstrap the **walk-forward OOS** trade returns (contiguous blocks, so loss-clusters / regime streaks are preserved — a plain i.i.d. shuffle understates drawdown). The spread answers: how much of the result is the lucky ordering, and how bad can drawdown plausibly get? `P(profitable)` near 50% means the OOS edge is indistinguishable from noise.

In [ ]:
mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"},
             title=f"OOS terminal-return distribution — {mc.n_sims} block-bootstrap sims").show()
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"},
             title="OOS max-drawdown distribution").show()

### Caveats

- **Window sizing is itself a hyperparameter.** If results only look good at one specific `TRAIN_BARS`/`TEST_BARS`, that's another overfit smell — vary them.
- **Block size** trades off preserving serial correlation (larger) against resampling diversity (smaller). `block=5` is a reasonable default for trade sequences.
- **Monte Carlo assumes the OOS trades are representative.** It quantifies luck *given* the edge; it cannot rescue a non-edge. That's why it runs on the walk-forward OOS trades, not the sweep winner.
- Swap `STRATEGY`/`GRID` for any strategy whose knobs live on `StrategyConfig` (the grid keys must be `StrategyConfig` fields).